In [ ]:
# GRU -- analisis de errores: los 137 falsos negativos de test con umbral 0.050
# (ver docs/gru/adr/0005-ajuste-umbral-decision-gru.md, seccion "Hallazgo: brecha de
# generalizacion val->test"). Este notebook no reentrena ni reajusta el umbral --
# investiga que comparten los ataques que el modelo, con ese umbral, sigue
# clasificando como benignos con alta confianza.

import sys
sys.path.append("../..")  # raíz del repo, para poder importar src.*

import numpy as np
import pandas as pd
import tensorflow as tf

from src.preprocessing.clean import load_and_clean, train_test_split_by_flow
from src.models.gru.error_analysis import (
    identificar_falsos_negativos,
    construir_metadata_secuencias,
    mapear_a_metadata,
)

np.random.seed(42)
tf.random.set_seed(42)

In [2]:
MODEL_PATH = "../../models/gru/baseline_w10.keras"
model = tf.keras.models.load_model(MODEL_PATH)

DATA_DIR = "../../data/sequences"
W = 10
UMBRAL_ELEGIDO = 0.050

X_test_w10 = np.load(f"{DATA_DIR}/X_test_w{W}.npy")
y_test_w10 = np.load(f"{DATA_DIR}/y_test_w{W}.npy")
y_proba_test = model.predict(X_test_w10, verbose=0).ravel()

print(f"X_test_w10: {X_test_w10.shape} | y_test_w10: {y_test_w10.shape}")
print(f"Umbral elegido (ver ADR 0005): {UMBRAL_ELEGIDO}")

X_test_w10: (67512, 10, 63) | y_test_w10: (67512,)
Umbral elegido (ver ADR 0005): 0.05


In [3]:
indices_fn = identificar_falsos_negativos(model, X_test_w10, y_test_w10, threshold=UMBRAL_ELEGIDO)
print(f"Falsos negativos encontrados: {len(indices_fn)} (esperado: 137, ver ADR 0005)")

Falsos negativos encontrados: 137 (esperado: 137, ver ADR 0005)


In [4]:
# Reconstrucción determinista del mismo X_test que produjo X_test_w10.npy, con
# Timestamp/bloque/Label intactos (ver el docstring de
# src/models/gru/error_analysis.py para el porqué de este mecanismo -- los .npy
# no guardan esa metadata, así que hay que reproducirla desde data/raw).
df = load_and_clean("../../data/raw")
df_ordenado = df.sort_values("Timestamp", kind="stable").reset_index(drop=True)
X_train, X_val, X_test, y_train, y_val, y_test = train_test_split_by_flow(df_ordenado)

metadata_secuencias = construir_metadata_secuencias(X_test, window_size=W)
print(f"metadata_secuencias: {metadata_secuencias.shape}")

AVISO: se encontraron 8 CSVs con Timestamp (parece el dataset completo de la semana). Se usa automáticamente el que corresponde al miércoles: ..\..\data\raw\GeneratedLabelledFlows\TrafficLabelling\Wednesday-workingHours.pcap_ISCX.csv
Cargado: ..\..\data\raw\GeneratedLabelledFlows\TrafficLabelling\Wednesday-workingHours.pcap_ISCX.csv
Etiquetas encontradas en el CSV: ['BENIGN', 'DoS slowloris', 'DoS Slowhttptest', 'DoS Hulk', 'DoS GoldenEye', 'Heartbleed']
Filas antes del filtro: 692703 | después (solo BENIGN + Low&Slow): 451326
Rango de Timestamp detectado: 2017-07-05 01:00:00 -> 2017-07-05 12:59:00
Valores nulos/infinitos detectados: 696
Filas duplicadas detectadas: 16
Columnas eliminadas: ['Flow ID', 'Source IP', 'Destination IP']
Columnas redundantes eliminadas: ['Subflow Fwd Packets', 'Fwd Header Length.1', 'Avg Fwd Segment Size', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Avg Bwd Segment Size', 'Subflow Bwd Bytes']
Filas con valores negativos anómalos eliminadas: 581 (0.1288%)
Co

In [5]:
assert len(metadata_secuencias) == len(y_test_w10) == 67512, (
    f"Longitud de metadata_secuencias ({len(metadata_secuencias)}) no coincide con "
    f"y_test_w10 ({len(y_test_w10)}) -- ¿cambió data/raw desde que se generaron los .npy?"
)
print("OK: metadata_secuencias coincide en longitud con y_test_w10 (67,512)")

OK: metadata_secuencias coincide en longitud con y_test_w10 (67,512)


In [6]:
# Metadata de TODOS los ataques de test (FN + TP), para poder comparar cada
# grupo contra el total y entre sí en las celdas siguientes.
indices_ataques = np.where(y_test_w10 == 1)[0]
indices_tp = np.setdiff1d(indices_ataques, indices_fn)

metadata_ataques = mapear_a_metadata(indices_ataques, X_test, df_ordenado, window_size=W)
casos_fn = metadata_ataques[metadata_ataques["posicion_secuencia"].isin(indices_fn)].reset_index(drop=True)

print(f"metadata_ataques: {metadata_ataques.shape} (total de ataques en test)")
print(f"casos_fn: {casos_fn.shape} (subconjunto de metadata_ataques)")
casos_fn.head()

metadata_ataques: (1103, 68) (total de ataques en test)
casos_fn: (137, 68) (subconjunto de metadata_ataques)


,posicion_secuencia,indice_original,bloque,Timestamp,Label,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,3056,241303,1,2017-07-05 09:01:00,DoS slowloris,53058,80,6,5001683,3,...,0,32,0.0,0.0,0.0,0.0,0.000,0.000,0.0,0.0
1,45320,309221,2,2017-07-05 10:09:00,DoS slowloris,33122,80,6,49407123,13,...,10,32,6409801.0,0.0,6409801.0,6409801.0,8342160.000,3901058.843,12800000.0,5778868.0
2,45321,309222,2,2017-07-05 10:09:00,DoS slowloris,33120,80,6,49408879,13,...,10,32,6410922.0,0.0,6410922.0,6410922.0,8342166.000,3901052.935,12800000.0,5778896.0
3,45322,309223,2,2017-07-05 10:09:00,DoS slowloris,33124,80,6,3000713,3,...,0,40,0.0,0.0,0.0,0.0,0.000,0.000,0.0,0.0
4,45323,309224,2,2017-07-05 10:09:00,DoS slowloris,33130,80,6,47542371,15,...,9,32,9409371.0,0.0,9409371.0,9409371.0,8342269.333,3901081.962,12800000.0,5778852.0


In [7]:
from src.models.gru.error_analysis import comparar_distribucion_label

tabla_label = comparar_distribucion_label(metadata_ataques, indices_fn)
print(tabla_label)

                  n_ataques_totales  n_falsos_negativos  \
Label                                                     
DoS slowloris                   914                 128   
DoS Slowhttptest                189                   9   

                  pct_fn_dentro_de_este_grupo  pct_del_total_de_ataques  \
Label                                                                     
DoS slowloris                           14.00                     82.86   
DoS Slowhttptest                         4.76                     17.14   

                  pct_del_total_de_fn  
Label                                  
DoS slowloris                   93.43  
DoS Slowhttptest                 6.57  


In [8]:
from src.models.gru.error_analysis import comparar_distribucion_bloque

tabla_bloque = comparar_distribucion_bloque(metadata_ataques, indices_fn)
print(tabla_bloque)

        n_ataques_totales  n_falsos_negativos  pct_fn_dentro_de_este_grupo  \
bloque                                                                       
2                     913                 127                        13.91   
4                     189                   9                         4.76   
1                       1                   1                       100.00   

        pct_del_total_de_ataques  pct_del_total_de_fn  
bloque                                                 
2                          82.77                92.70  
4                          17.14                 6.57  
1                           0.09                 0.73  


In [9]:
from src.models.gru.error_analysis import distribuir_probabilidad_fn

tabla_probabilidad = distribuir_probabilidad_fn(y_proba_test, indices_fn)
print(tabla_probabilidad)

                 n_falsos_negativos  pct_del_total_de_fn
(-0.001, 0.001]                  21                15.33
(0.001, 0.01]                    95                69.34
(0.01, 0.03]                     15                10.95
(0.03, 0.05]                      6                 4.38


In [10]:
from src.models.gru.error_analysis import comparar_features_crudas

columnas_a_comparar = ["Flow Duration", "Flow Bytes/s", "Flow Packets/s", "Flow IAT Mean"]
tabla_features = comparar_features_crudas(metadata_ataques, indices_fn, indices_tp, columnas_a_comparar)
print(tabla_features)

                media_falsos_negativos  mediana_falsos_negativos  \
Flow Duration             1.095026e+07               9903.000000   
Flow Bytes/s              2.065773e+07              48773.099060   
Flow Packets/s            2.128214e+04                302.938504   
Flow IAT Mean             1.904061e+06               4951.500000   

                media_verdaderos_positivos  mediana_verdaderos_positivos  
Flow Duration                 3.916498e+07                  4.082887e+06  
Flow Bytes/s                  5.081825e+07                  2.412543e+01  
Flow Packets/s                5.397709e+04                  9.250490e-01  
Flow IAT Mean                 1.006358e+07                  1.501974e+06  
